# Urdu chunking: lower-size dense grid (10/25/40/50)

This notebook runs the missing experiment from `docs/audit/revised_experiments.md`.

**Before running**

1. In Colab: Runtime -> Change runtime type -> GPU (T4 is enough; A100 is faster).
2. Upload these two files to a Drive folder, e.g. `MyDrive/urdu_chunking/urdu_chunking_full/`:
   - `corpus_ur.jsonl`
   - `questions.csv`
   The code is cloned from the public repository below; do not upload it.
3. Set `DATA` below if the Drive folder is different.

The script is resumable: if Colab disconnects, rerun the clone/path cells and the run cell; it skips
runs already in `small_sizes/runs.jsonl`. Do not publish the data folder; it contains UQA text.


In [ ]:
!pip -q install sentence-transformers==6.2.0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
REPO = 'https://github.com/devxyasir/urdu-chunking-strategy.git'
ART = '/content/urdu-chunking-artifact'
if not os.path.isdir(ART):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, ART], check=True)
else:
    subprocess.run(['git', '-C', ART, 'pull', '--ff-only'], check=True)
print('code at', ART)

In [ ]:
PKG = '/content/drive/MyDrive/urdu_chunking'   # <-- change if needed
DATA = f'{PKG}/urdu_chunking_full'
CODE = '/content/urdu-chunking-artifact/code'
OUT = f'{PKG}/small_sizes'
import os
for p in [DATA + '/corpus_ur.jsonl', DATA + '/questions.csv', CODE + '/run_small_sizes.py',
          CODE + '/chunking.py', CODE + '/metrics.py']:
    print(('OK  ' if os.path.exists(p) else 'MISS'), p)

In [ ]:
import os
os.environ['DATA'] = DATA
os.environ['OUT'] = OUT
os.chdir(CODE)
!python run_small_sizes.py --data-dir "$DATA" --out-dir "$OUT" \
  --models bge e5 qwen labse --sizes 10 25 40 50 --device cuda

In [ ]:
import json, pandas as pd
rows = [json.loads(l) for l in open(f'{OUT}/runs.jsonl', encoding='utf-8')]
df = pd.DataFrame([{**{'model': r['model'].split('/')[-1], 'size': r['size'],
                       'chunks': r['chunks']}, **r['metrics']} for r in rows])
df = df.sort_values(['model', 'size'])
display(df[['model', 'size', 'chunks', 'bm500x', 'bm500x_lo', 'bm500x_hi', 'bm500', 'answer_split_rate']])

In [ ]:
import matplotlib.pyplot as plt
for model, g in df.groupby('model'):
    g = g.sort_values('size')
    plt.plot(g['size'], g['bm500x'], marker='o', label=model)
plt.xlabel('chunk size (words)'); plt.ylabel('BM-Recall@500 (crossing)')
plt.legend(); plt.grid(alpha=.3); plt.show()

## Decision rule (pre-registered)

- If 40 or 25 beats 50 by at least 1.0 point with a paired 95% CI excluding zero, replace the
  paper's argmax and rerun the boundary discussion at that size.
- If 50 stays the argmax with paired CIs entirely below zero, the paper can say "50 is the best
  tested size and the lower bound of the tested range"; still not an interior optimum.
- If differences are within +/-1 point with CIs crossing zero, report a plateau.

Do not add 30/35/45 after seeing the test results unless the pre-registered rule requires a
follow-up. That would be sequential tuning on the test set.
